# 08 — Recover Published Post Registry

Reconstructs `content_queue.json` published entries from Mastodon and Bluesky APIs.
Use this when the S3 published registry is lost or corrupted.

**Workflow:**
1. Run cells 1–4 (read-only) — fetch + preview recovered records
2. Inspect the preview table carefully
3. Run cell 5 — you will be prompted to type **confirm** before anything is written to S3

In [1]:
import sys
sys.path.insert(0, '..')

import os
import re
import uuid
from datetime import datetime, timezone

from dotenv import load_dotenv
load_dotenv('../.env', override=True)

from agent.models import ContentQueue, Draft
from agent.platforms.mastodon import MastodonClient
from agent.platforms.bluesky import BlueskyClient
from agent.storage import S3Storage, load_model

# Target the PROD prefix — this is where the registry lives
store = S3Storage(
    bucket=os.environ['S3_BUCKET'],
    prefix=os.environ['S3_STATE_PREFIX_PROD'],
    access_key=os.environ['SCW_ACCESS_KEY'],
    secret_key=os.environ['SCW_SECRET_KEY'],
)

BLOG_URL_PATTERN = re.compile(r'https?://(?:www\.)?fretchen\.eu[^\s"<>]*')

def parse_link(text: str) -> str | None:
    """Extract the first fretchen.eu URL from post text, strip UTM params."""
    m = BLOG_URL_PATTERN.search(text or '')
    if not m:
        return None
    url = m.group(0).rstrip('.,)')
    # Strip UTM query string
    return url.split('?')[0]

print('Setup complete')
print(f'S3 prefix: {store.prefix}')

Setup complete
S3 prefix: growth-agent/


## 1. Fetch from Mastodon

Paginates your timeline (up to 3 pages × 40 posts = 120 posts max).
Filters to posts where `application.name == 'growth-agent'` when the field is present.

In [ ]:
import httpx

MASTODON_INSTANCE = os.environ.get('MASTODON_INSTANCE', 'https://mastodon.social')
MASTODON_TOKEN = os.environ['MASTODON_ACCESS_TOKEN']

mastodon_headers = {'Authorization': f'Bearer {MASTODON_TOKEN}'}
base = MASTODON_INSTANCE.rstrip('/')

# Get account id
me = httpx.get(f'{base}/api/v1/accounts/verify_credentials', headers=mastodon_headers, timeout=15)
me.raise_for_status()
account_id = me.json()['id']
print(f'Account id: {account_id}  (@{me.json()["acct"]})')

# Paginate up to 3 pages
mastodon_posts = []
max_id = None
for page in range(3):
    params = {'limit': 40}
    if max_id:
        params['max_id'] = max_id
    r = httpx.get(f'{base}/api/v1/accounts/{account_id}/statuses',
                  headers=mastodon_headers, params=params, timeout=15)
    r.raise_for_status()
    page_posts = r.json()
    if not page_posts:
        break
    mastodon_posts.extend(page_posts)
    max_id = page_posts[-1]['id']

print(f'Fetched {len(mastodon_posts)} Mastodon statuses')

# Filter: keep only posts from growth-agent app OR containing a fretchen.eu link.
# Search raw HTML — Mastodon wraps URLs in <span class="invisible/ellipsis"> which
# fragments the URL when tags are stripped. parse_link works on raw HTML because
# BLOG_URL_PATTERN stops at " so it reads the full URL from href="...".
def is_agent_post(s: dict) -> bool:
    app_name = (s.get('application') or {}).get('name', '')
    if app_name == 'growth-agent':
        return True
    return bool(parse_link(s.get('content', '')))

mastodon_agent_posts = [s for s in mastodon_posts if is_agent_post(s)]
print(f'After filter: {len(mastodon_agent_posts)} posts look like growth-agent posts')

## 2. Fetch from Bluesky

Paginates your author feed (up to 3 pages × 40 posts = 120 posts max).
No app-tag available — keeps all posts containing a `fretchen.eu` link.
You can deselect false positives in the preview step.

In [3]:
bsky = BlueskyClient(
    handle=os.environ['BLUESKY_HANDLE'],
    app_password=os.environ['BLUESKY_APP_PASSWORD'],
)

bluesky_posts = []
cursor = None
for page in range(3):
    params = {'actor': bsky.did, 'limit': 40}
    if cursor:
        params['cursor'] = cursor
    r = bsky.client.get(f'{BlueskyClient.BASE_URL}/app.bsky.feed.getAuthorFeed',
                        params=params, timeout=15)
    r.raise_for_status()
    data = r.json()
    feed = data.get('feed', [])
    if not feed:
        break
    bluesky_posts.extend(feed)
    cursor = data.get('cursor')
    if not cursor:
        break

bsky.close()
print(f'Fetched {len(bluesky_posts)} Bluesky posts')

# Keep only posts containing a fretchen.eu link
bluesky_agent_posts = [
    item for item in bluesky_posts
    if parse_link(item.get('post', {}).get('record', {}).get('text', ''))
]
print(f'After filter: {len(bluesky_agent_posts)} posts contain a fretchen.eu link')

Fetched 76 Bluesky posts
After filter: 23 posts contain a fretchen.eu link


## 3. Build + Preview Recovered Records

Inspect this table carefully before proceeding to cell 5.
Remove any false positives by editing `recovered_drafts` if needed.

In [ ]:
from html import unescape

recovered_drafts: list[Draft] = []

# --- Mastodon ---
for s in mastodon_agent_posts:
    plain = unescape(re.sub(r'<[^>]+>', ' ', s.get('content', ''))).strip()
    # Extract link from raw HTML — tag-stripping fragments Mastodon's span-wrapped URLs
    link = parse_link(s.get('content', ''))
    published_at = datetime.fromisoformat(s['created_at'].replace('Z', '+00:00'))
    draft_id = f"recovered_mastodon_{published_at.strftime('%Y%m%d%H%M%S')}"
    recovered_drafts.append(Draft(
        id=draft_id,
        channel='mastodon',
        language=s.get('language') or 'en',
        content=plain,
        link=link,
        status='published',
        scheduled_at=published_at,
        published_at=published_at,
        platform_id=s['id'],
    ))

# --- Bluesky ---
for item in bluesky_agent_posts:
    record = item.get('post', {}).get('record', {})
    text = record.get('text', '')
    link = parse_link(text)
    created_at_raw = record.get('createdAt', '')
    published_at = datetime.fromisoformat(created_at_raw.replace('Z', '+00:00'))
    draft_id = f"recovered_bluesky_{published_at.strftime('%Y%m%d%H%M%S')}"
    recovered_drafts.append(Draft(
        id=draft_id,
        channel='bluesky',
        language='en',
        content=text,
        link=link,
        status='published',
        scheduled_at=published_at,
        published_at=published_at,
        platform_id=item.get('post', {}).get('uri'),
    ))

recovered_drafts.sort(key=lambda d: d.published_at or datetime.min.replace(tzinfo=timezone.utc))

print(f'Recovered {len(recovered_drafts)} total drafts\n')
print(f'{"#":<3} {"channel":<10} {"published_at":<22} {"platform_id":<45} {"link":<45} {"content preview"}')
print('-' * 150)
for i, d in enumerate(recovered_drafts):
    ts = d.published_at.strftime('%Y-%m-%d %H:%M') if d.published_at else 'unknown'
    preview = (d.content or '')[:60].replace('\n', ' ')
    print(f'{i:<3} {d.channel:<10} {ts:<22} {(d.platform_id or "(none)"):<45} {(d.link or "(no link)"):<45} {preview}')

## 4. Merge into S3

You will be prompted to type **confirm** before anything is written to S3.
Typing anything else (or pressing Enter with no input) cancels without writing.

Deduplication: existing published entries with the same `id` or same
`(channel, published_at)` pair are not duplicated.

In [5]:
confirm = input("Type 'confirm' to write to S3 (anything else cancels): ")
if confirm.strip().lower() != "confirm":
    raise RuntimeError("Cancelled — nothing written.")

queue = load_model(store, 'content_queue.json', ContentQueue)
existing_before = len(queue.published)

# Build dedup keys from existing published entries
existing_ids = {d.id for d in queue.published}
existing_keys = {
    (d.channel, d.published_at.replace(microsecond=0) if d.published_at else None)
    for d in queue.published
}

added = 0
for d in recovered_drafts:
    if d.id in existing_ids:
        continue
    key = (d.channel, d.published_at.replace(microsecond=0) if d.published_at else None)
    if key in existing_keys:
        continue
    queue.published.append(d)
    existing_ids.add(d.id)
    existing_keys.add(key)
    added += 1

store.write('content_queue.json', queue)

print(f'Published before : {existing_before}')
print(f'Added            : {added}')
print(f'Published after  : {len(queue.published)}')
print('\nDone. Run: uv run python run_local.py --diagnose --prod  to verify.')

RuntimeError: Cancelled — nothing written.